6mins 11.9 secs

## Libraries

In [1]:
import numpy as np
import pandas as pd
import time
import random

from sklearn.model_selection import ParameterGrid

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.neighbors import NearestNeighbors

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_

In [2]:
print("Torch version:", torch.__version__)
print("CUDA (in torch):", torch.version.cuda)
print("cuda.is_available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))

Torch version: 2.10.0.dev20251203+cu128
CUDA (in torch): 12.8
cuda.is_available: True
device count: 1
device: NVIDIA GeForce RTX 5070 Ti
capability: (12, 0)


## Config

In [3]:
TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TUNE_TRAIN_END = pd.Timestamp("2021-03-31")   # train for hyperparam tuning
TUNE_VAL_END   = pd.Timestamp("2022-03-31")   # validation period for tuning

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- feature lists (adjust to match your data) ----
continuous_cols = [
    "AverageNeighbourPrice",
    "local_I",
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
]

categorical_cols = [
    "LMIQuadrant__2",
    "LMIQuadrant__3",
    "LMIQuadrant__4",
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

feature_cols = continuous_cols + categorical_cols

# random search settings
N_RANDOM_CONFIGS = 12   # including baseline
MAX_EPOCHS       = 50
PATIENCE         = 6    # epochs without improvement
BATCH_SIZE       = 32
MAX_GRAD_NORM    = 5.0  # gradient clipping

# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)



## Load data

In [4]:
# tuning subset (only needed up to TUNE_VAL_END)
df = pd.read_excel("../../data/full_data.xlsx", parse_dates=[TIME_COL])
df_full = df.copy()
df = df.sort_values([TIME_COL, ENTITY_COL]).reset_index(drop=True)
df = df[df[TIME_COL] <= TUNE_VAL_END].copy()
df = df[df[TIME_COL] >= pd.Timestamp("2007-04-01")].copy()

print("Tuning df date range:", df[TIME_COL].min(), "→", df[TIME_COL].max())

# initial node set from tuning data
la_order = sorted(df[ENTITY_COL].unique())
print("LAs in tuning df:", len(la_order))

# full data for robust centroids
df_full = df_full.sort_values([ENTITY_COL, TIME_COL]).reset_index(drop=True)

# forward/backward fill centroids per LA
df_full[["centroid_x", "centroid_y"]] = (
    df_full.groupby(ENTITY_COL)[["centroid_x", "centroid_y"]]
           .ffill()
           .bfill()
)

centroid_df = (
    df_full.drop_duplicates(ENTITY_COL)
           .set_index(ENTITY_COL)[["centroid_x", "centroid_y"]]
)

# align centroids to LAs in tuning data
centroid_df = centroid_df.loc[la_order]

print("NaNs in centroid_df before drop:")
print(centroid_df.isna().sum())

bad_las = centroid_df[centroid_df.isna().any(axis=1)].index.tolist()
if bad_las:
    print(f"⚠ Dropping {len(bad_las)} LAs with missing centroids:", bad_las)
    centroid_df = centroid_df.dropna()
    la_order = centroid_df.index.tolist()
    df = df[df[ENTITY_COL].isin(la_order)].copy()

N = len(la_order)
centroids = centroid_df.loc[la_order].values.astype(np.float32)

print("Final LAs used:", N)

Tuning df date range: 2007-04-01 00:00:00 → 2022-03-01 00:00:00
LAs in tuning df: 294
NaNs in centroid_df before drop:
centroid_x    0
centroid_y    0
dtype: int64
Final LAs used: 294


## Node order and adjacency

In [5]:
# # use K=8 neighbours to avoid low degree
# nbrs = NearestNeighbors(n_neighbors=8).fit(centroids)
# _, idx = nbrs.kneighbors(centroids)

# A = np.zeros((N, N), dtype=np.float32)
# for i in range(N):
#     for j in idx[i][1:]:   # skip self
#         A[i, j] = 1.0
#         A[j, i] = 1.0

# # add self-loops and normalise
# A = A + np.eye(N, dtype=np.float32)
# deg = A.sum(axis=1)
# print("Min/Max degree after KNN+I:", deg.min(), deg.max())

# D_inv_sqrt = np.diag(1.0 / np.sqrt(deg + 1e-8))
# A_hat_np = D_inv_sqrt @ A @ D_inv_sqrt
# A_hat = torch.tensor(A_hat_np, dtype=torch.float32, device=DEVICE)

# print("A_hat shape:", A_hat.shape)

## Complete panel [T, N] and build X_all, y_all (NO NaNs)

In [6]:
# ensure 1 row per (Date, AreaCode) before panel completion
df = (
    df.sort_values([TIME_COL, ENTITY_COL])
      .drop_duplicates(subset=[TIME_COL, ENTITY_COL], keep="last")
      .copy()
)

dates = pd.Index(sorted(df[TIME_COL].unique()))
T_total = len(dates)
print("Total time steps (T_total):", T_total)

full_index = pd.MultiIndex.from_product(
    [dates, la_order],
    names=[TIME_COL, ENTITY_COL]
)

df_panel = (
    df.set_index([TIME_COL, ENTITY_COL])
      .reindex(full_index)
      .sort_index()
)

# forward/backward fill per LA
df_panel[feature_cols + [TARGET_COL]] = (
    df_panel[feature_cols + [TARGET_COL]]
        .groupby(level=ENTITY_COL)
        .ffill()
        .bfill()
)

# =========================================================
# ADD LAGGED PRICE FEATURES (1 and 12 months)
# =========================================================
df_panel["price_lag1"] = (
    df_panel
    .groupby(level=ENTITY_COL)[TARGET_COL]
    .shift(1)
)

df_panel["price_lag12"] = (
    df_panel
    .groupby(level=ENTITY_COL)[TARGET_COL]
    .shift(12)
)

# fill lags within each LA (so we don't introduce NaNs)
df_panel[["price_lag1", "price_lag12"]] = (
    df_panel[["price_lag1", "price_lag12"]]
        .groupby(level=ENTITY_COL)
        .ffill()
        .bfill()
)

# extend feature set to include lagged prices
lag_price_cols = ["price_lag1", "price_lag12"]
feature_cols = feature_cols + lag_price_cols

missing_total = df_panel[feature_cols + [TARGET_COL]].isna().sum().sum()
if missing_total > 0:
    print(f"⚠ {missing_total} NaNs after ffill/bfill, filling with column means.")
    col_means = df_panel[feature_cols + [TARGET_COL]].mean()
    df_panel[feature_cols + [TARGET_COL]] = df_panel[feature_cols + [TARGET_COL]].fillna(col_means)

print("NaNs after panel completion:",
      df_panel[feature_cols + [TARGET_COL]].isna().sum().sum())

# convert to tensors [T, N, F], [T, N]
F = len(feature_cols)
X_all = (
    df_panel[feature_cols]
    .to_numpy(dtype=np.float32)
    .reshape(T_total, N, F)
)
y_all = (
    df_panel[TARGET_COL]
    .to_numpy(dtype=np.float32)
    .reshape(T_total, N)
)

print("X_all shape:", X_all.shape)
print("y_all shape:", y_all.shape)

Total time steps (T_total): 180
⚠ 1 NaNs after ffill/bfill, filling with column means.
NaNs after panel completion: 0
X_all shape: (180, 294, 31)
y_all shape: (180, 294)


## Train val indices

In [7]:
train_end_idx = np.searchsorted(dates, TUNE_TRAIN_END, side="right")
val_end_idx   = np.searchsorted(dates, TUNE_VAL_END,   side="right")

print(f"Train ends at idx {train_end_idx-1}, date {dates[train_end_idx-1].date()}")
print(f"Val   ends at idx {val_end_idx-1}, date {dates[val_end_idx-1].date()}")


Train ends at idx 167, date 2021-03-01
Val   ends at idx 179, date 2022-03-01


## Correlation-Based KNN adjacency

In [8]:
# use only training period for correlation to avoid peeking into the future
y_train_for_corr = y_all[:train_end_idx]   # shape [T_train, N]

# compute N x N correlation matrix across LAs
# rows = time, columns = LAs -> transpose so variables are rows
corr = np.corrcoef(y_train_for_corr.T)     # shape [N, N]

# replace any NaNs (e.g. constant series) with 0 correlation
corr = np.nan_to_num(corr, nan=0.0)

# we don't want self-correlation to drive KNN
np.fill_diagonal(corr, 0.0)

# choose how many neighbours per node
K = 8  # you can tune this (e.g. 5, 8, 10)

A = np.zeros((N, N), dtype=np.float32)

for i in range(N):
    # sort neighbours by absolute correlation (strongest relationships first)
    nbr_idx = np.argsort(-np.abs(corr[i]))[:K]  # top-K indices

    for j in nbr_idx:
        A[i, j] = 1.0
        A[j, i] = 1.0   # make the graph undirected

# add self-loops
A = A + np.eye(N, dtype=np.float32)

# normalise adjacency: A_hat = D^{-1/2} A D^{-1/2}
deg = A.sum(axis=1)
print("Min/Max degree (correlation graph):", deg.min(), deg.max())

D_inv_sqrt = np.diag(1.0 / np.sqrt(deg + 1e-8))
A_hat_np = D_inv_sqrt @ A @ D_inv_sqrt

A_hat = torch.tensor(A_hat_np, dtype=torch.float32, device=DEVICE)
print("Correlation-based A_hat shape:", A_hat.shape)

Min/Max degree (correlation graph): 9.0 47.0
Correlation-based A_hat shape: torch.Size([294, 294])


## Scale features and target

In [9]:
# X: StandardScaler
X_train_flat = X_all[:train_end_idx].reshape(-1, F)

x_scaler = StandardScaler()
X_all_scaled = X_all.copy()
X_all_scaled[:train_end_idx] = x_scaler.fit_transform(X_train_flat).reshape(-1, N, F)
X_all_scaled[train_end_idx:val_end_idx] = x_scaler.transform(
    X_all[train_end_idx:val_end_idx].reshape(-1, F)
).reshape(-1, N, F)

# y: RobustScaler
y_train_flat = y_all[:train_end_idx].reshape(-1, 1)

y_scaler = RobustScaler()
y_all_scaled = y_all.copy()
y_all_scaled[:train_end_idx] = y_scaler.fit_transform(y_train_flat).reshape(-1, N)
y_all_scaled[train_end_idx:val_end_idx] = y_scaler.transform(
    y_all[train_end_idx:val_end_idx].reshape(-1, 1)
).reshape(-1, N)

y_scale_factor = float(y_scaler.scale_[0])

print("NaNs in X_all_scaled:", np.isnan(X_all_scaled).sum())
print("NaNs in y_all_scaled:", np.isnan(y_all_scaled).sum())

NaNs in X_all_scaled: 0
NaNs in y_all_scaled: 0


## Dataset class (window will vary per config)

In [10]:
class SpatioTemporalDataset(Dataset):
    def __init__(self, X, y, start_t, end_t, window):
        """
        X: [T, N, F], y: [T, N]
        target at t in [start_t, end_t), input seq [t-window, ..., t-1]
        """
        self.X = X
        self.y = y
        self.window = window
        self.indices = [
            t for t in range(start_t, end_t)
            if t - window >= 0
        ]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t = self.indices[idx]
        X_seq = self.X[t - self.window:t]   # [window, N, F]
        y_t   = self.y[t]                  # [N]
        return (
            torch.tensor(X_seq, dtype=torch.float32),
            torch.tensor(y_t,   dtype=torch.float32),
        )

## Model

In [11]:
class GraphConv(nn.Module):
    def __init__(self, in_feats, out_feats):
        super().__init__()
        self.linear = nn.Linear(in_feats, out_feats)

    def forward(self, X, A_hat):
        # X: [B, N, F], A_hat: [N, N]
        return self.linear(torch.einsum("ij,bjf->bif", A_hat, X))

class TGCNCell(nn.Module):
    def __init__(self, in_feats, hidden_dim, dropout=0.0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.gc_zr = GraphConv(in_feats + hidden_dim, 2 * hidden_dim)
        self.gc_h  = GraphConv(in_feats + hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X_t, H_prev, A_hat):
        if H_prev is None:
            H_prev = torch.zeros(
                X_t.size(0), X_t.size(1), self.hidden_dim,
                device=X_t.device
            )
        XH = torch.cat([X_t, H_prev], dim=-1)  # [B, N, F+H]

        ZR = torch.sigmoid(self.gc_zr(XH, A_hat))  # [B, N, 2H]
        Z, R = torch.chunk(ZR, 2, dim=-1)

        XH_candidate = torch.cat([X_t, R * H_prev], dim=-1)
        H_tilde = torch.tanh(self.gc_h(XH_candidate, A_hat))

        H_new = (1 - Z) * H_prev + Z * H_tilde
        H_new = self.dropout(H_new)
        return H_new

class TGCN(nn.Module):
    def __init__(self, num_nodes, in_feats, hidden_dim, dropout=0.0):
        super().__init__()
        self.cell = TGCNCell(in_feats, hidden_dim, dropout=dropout)
        self.out  = nn.Linear(hidden_dim, 1)

    def forward(self, X_seq, A_hat):
        # X_seq: [B, T, N, F]
        B, T, N, F = X_seq.shape
        H = None
        for t in range(T):
            X_t = X_seq[:, t]          # [B, N, F]
            H   = self.cell(X_t, H, A_hat)
        y_hat = self.out(H).squeeze(-1)  # [B, N]
        return y_hat


## Hyperparameter space and random configs

In [12]:
param_space = {
    "WINDOW":      [12, 24],
    "HIDDEN_DIM":  [32, 64, 128],
    "DROPOUT":     [0.0, 0.3],
    "LR":          [1e-3, 5e-4],
    "WEIGHT_DECAY":[0.0, 1e-4],
}

# full grid over all combinations
param_grid = list(ParameterGrid(param_space))
print(f"Number of configs in full grid: {len(param_grid)}")

# if you still want to guarantee the baseline is included (it is, but just in case):
baseline_config = {
    "WINDOW": 24,
    "HIDDEN_DIM": 32,
    "DROPOUT": 0.3,
    "LR": 1e-3,
    "WEIGHT_DECAY": 1e-4,
}
# make sure it's in the grid (usually it already will be)
if baseline_config not in param_grid:
    param_grid.append(baseline_config)

configs_to_run = param_grid


Number of configs in full grid: 48


## Train and Eval with early stopping

In [13]:
mse_loss = nn.MSELoss()
results = []

for cfg_id, cfg in enumerate(configs_to_run, start=1):
    WINDOW     = cfg["WINDOW"]
    HIDDEN_DIM = cfg["HIDDEN_DIM"]
    DROPOUT    = cfg["DROPOUT"]
    LR         = cfg["LR"]
    WD         = cfg["WEIGHT_DECAY"]

    print(f"\n=== Config {cfg_id}/{len(configs_to_run)} ===")
    print(cfg)

    # build datasets
    train_ds = SpatioTemporalDataset(
        X_all_scaled, y_all_scaled,
        start_t=WINDOW, end_t=train_end_idx,
        window=WINDOW
    )
    val_ds = SpatioTemporalDataset(
        X_all_scaled, y_all_scaled,
        start_t=train_end_idx, end_t=val_end_idx,
        window=WINDOW
    )

    if len(train_ds) == 0 or len(val_ds) == 0:
        print("  (skip: insufficient data for this WINDOW)")
        continue

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = TGCN(num_nodes=N, in_feats=F, hidden_dim=HIDDEN_DIM, dropout=DROPOUT).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)

    best_val_mse = np.inf
    best_epoch   = -1
    epochs_no_improve = 0
    best_state = None
    config_valid = True

    start_time = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        batch_losses = []

        for X_seq, y_t in train_loader:
            X_seq = X_seq.to(DEVICE)
            y_t   = y_t.to(DEVICE)

            optimizer.zero_grad()
            y_hat = model(X_seq, A_hat)
            loss  = mse_loss(y_hat, y_t)

            if not torch.isfinite(loss):
                print(f"  ⚠ Non-finite training loss at epoch {epoch}. Aborting this config.")
                config_valid = False
                break

            loss.backward()
            clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()
            batch_losses.append(loss.item())

        if not config_valid:
            break

        # validation
        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_seq, y_t in val_loader:
                X_seq = X_seq.to(DEVICE)
                y_t   = y_t.to(DEVICE)
                y_hat = model(X_seq, A_hat)
                val_loss = mse_loss(y_hat, y_t)
                if torch.isfinite(val_loss):
                    val_losses.append(val_loss.item())

        if len(val_losses) == 0:
            print("  ⚠ All validation losses were non-finite. Skipping this config.")
            config_valid = False
            break

        val_mse = float(np.mean(val_losses))
        val_rmse_orig = np.sqrt(val_mse) * y_scale_factor

        print(f"  Epoch {epoch:03d} | "
              f"train MSE={np.mean(batch_losses):.4f} | "
              f"val MSE={val_mse:.4f} | "
              f"val RMSE(£)={val_rmse_orig:,.1f}")

        if val_mse + 1e-6 < best_val_mse:
            best_val_mse = val_mse
            best_epoch   = epoch
            epochs_no_improve = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"  Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs).")
                break

    total_time = time.time() - start_time

    if not config_valid or best_epoch == -1:
        print("  ❌ Config failed. Not logging metrics.")
        continue

    best_val_rmse_orig = np.sqrt(best_val_mse) * y_scale_factor

    results.append({
        "model_type": "TGCN",
        "WINDOW": WINDOW,
        "HIDDEN_DIM": HIDDEN_DIM,
        "DROPOUT": DROPOUT,
        "LR": LR,
        "WEIGHT_DECAY": WD,
        "best_val_MSE_scaled": best_val_mse,
        "best_val_RMSE_orig": best_val_rmse_orig,
        "best_epoch": best_epoch,
        "epochs_run": min(epoch, MAX_EPOCHS),
        "train_time_sec": total_time,
    })


=== Config 1/48 ===
{'DROPOUT': 0.0, 'HIDDEN_DIM': 32, 'LR': 0.001, 'WEIGHT_DECAY': 0.0, 'WINDOW': 12}
  Epoch 001 | train MSE=1.0701 | val MSE=1.7230 | val RMSE(£)=173,997.8
  Epoch 002 | train MSE=0.8040 | val MSE=1.3929 | val RMSE(£)=156,444.5
  Epoch 003 | train MSE=0.6262 | val MSE=1.1344 | val RMSE(£)=141,185.1
  Epoch 004 | train MSE=0.5089 | val MSE=0.9404 | val RMSE(£)=128,546.8
  Epoch 005 | train MSE=0.4407 | val MSE=0.7917 | val RMSE(£)=117,943.0
  Epoch 006 | train MSE=0.3915 | val MSE=0.6737 | val RMSE(£)=108,799.7
  Epoch 007 | train MSE=0.3498 | val MSE=0.5813 | val RMSE(£)=101,067.0
  Epoch 008 | train MSE=0.3116 | val MSE=0.5158 | val RMSE(£)=95,197.3
  Epoch 009 | train MSE=0.2841 | val MSE=0.4711 | val RMSE(£)=90,981.1
  Epoch 010 | train MSE=0.2660 | val MSE=0.4385 | val RMSE(£)=87,782.7
  Epoch 011 | train MSE=0.2531 | val MSE=0.4129 | val RMSE(£)=85,173.9
  Epoch 012 | train MSE=0.2427 | val MSE=0.3929 | val RMSE(£)=83,091.9
  Epoch 013 | train MSE=0.2325 | val 

## Results

In [15]:
results_df = pd.DataFrame(results).sort_values("best_val_RMSE_orig").reset_index(drop=True)
print("\n=== TOP CONFIGS BY VAL RMSE (ORIGINAL SCALE) ===")
print(results_df.head(10))

results_df.to_csv("../../results/tgcn_random_search_tuning_pre2022_with_lags_1_12_avg_neighbour_price_correlation.csv", index=False)
print("\nSaved tuning results to ../../results/tgcn_random_search_tuning_pre2022_with_lags_1_12_avg_neighbour_price_correlation.csv")


=== TOP CONFIGS BY VAL RMSE (ORIGINAL SCALE) ===
  model_type  WINDOW  HIDDEN_DIM  DROPOUT     LR  WEIGHT_DECAY  \
0       TGCN      24         128      0.0  0.001        0.0000   
1       TGCN      12         128      0.0  0.001        0.0000   
2       TGCN      24         128      0.0  0.001        0.0001   
3       TGCN      12         128      0.0  0.001        0.0001   
4       TGCN      12          64      0.0  0.001        0.0001   
5       TGCN      24          64      0.0  0.001        0.0000   
6       TGCN      24         128      0.3  0.001        0.0001   
7       TGCN      12         128      0.3  0.001        0.0001   
8       TGCN      24          64      0.0  0.001        0.0001   
9       TGCN      12         128      0.3  0.001        0.0000   

   best_val_MSE_scaled  best_val_RMSE_orig  best_epoch  epochs_run  \
0             0.140829        49744.883602          50          50   
1             0.148260        51040.434807          50          50   
2            